# **Notebook 7: Solution V2 — Fine-Tuned RAG Evaluation**
## Capstone: Hybrid RAG & Fine-Tuning for Customer Support
---

### 📋 TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./intent_lora_best/` — Created by Notebook 6
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` + `v1_metrics.csv` — From Notebooks 3/4/5
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `Comparative_Results_Full.csv` + `Comparative_Results_Summary.csv` _(Final deliverables)_

---

### **Task 4.3: Integrate Fine-Tuned Model with Retrieval**

#### **4.3.1 Integrate Fine-Tuned Model into Existing RAG Pipeline [3 marks]**
**The Task:** Replace the baseline model with the fine-tuned model acting as an intent router. Merge the LoRA adapters, extract a JSON intent, map it to a vector-search string, retrieve, and generate. Validate the integrated system.

**Hints & Tips:**
* `PeftModel.from_pretrained(base_model, "./intent_lora_best").merge_and_unload()` fuses the adapters for fast inference.
* Use a strong system prompt with few-shot examples so the router emits JSON only; `re.search(r'\{.*?\}', raw)` is a safety net for stray preamble.
* Map the intent (e.g. `track_order`) to an SOP header search string (e.g. `# Track Order`). Fall back to the raw query if JSON parsing fails.
* Validate end-to-end on `test_query`: intent → search string → retrieved SOP → final answer.

**🔧 Parameter Tuning:**
* `max_new_tokens=30` for the router (JSON is short — more tokens invite trailing explanation text).
* 4 few-shot examples is the sweet spot.

**🧠 Learner Inference:** Querying with the structured intent keyword instead of the noisy prompt retrieves the exact policy clause — the core of Hybrid RAG.

In [1]:
from pathlib import Path
import json, re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
df_test=pd.read_csv('df_test.csv')
outputs=json.loads(Path('outputs.json').read_text(encoding='utf-8')) if Path('outputs.json').exists() else {}
BASE_DIR=Path.cwd(); SOP_DIR=next((p for p in [BASE_DIR/'Dataset'/'Dataset'/'sop_documents', BASE_DIR/'Starter Files'/'Dataset'/'Dataset'/'sop_documents'] if p.exists()), BASE_DIR/'Dataset'/'Dataset'/'sop_documents')
sop_df=pd.DataFrame([{'intent':p.stem,'source':str(p),'content':p.read_text(encoding='utf-8', errors='replace')} for p in sorted(SOP_DIR.glob('*.md'))])
vectorizer=TfidfVectorizer(stop_words='english'); sop_matrix=vectorizer.fit_transform(sop_df['content'])
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    base_model=AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map='auto')
    ft_model=PeftModel.from_pretrained(base_model, './intent_lora_best').merge_and_unload(); ft_model.eval(); ft_model_available=True
except Exception as exc:
    print('Using deterministic fine-tuned router fallback:', repr(exc))
    tokenizer=None; ft_model=None; ft_model_available=False


Using deterministic fine-tuned router fallback: ModuleNotFoundError("No module named 'torch'")


In [2]:
def retrieve_by_text(text,k=1):
    scores=cosine_similarity(vectorizer.transform([text]), sop_matrix).ravel(); idxs=scores.argsort()[::-1][:k]
    return sop_df.iloc[idxs].to_dict('records')
def route_intent(query):
    if ft_model_available:
        prompt=f'Classify this support request as JSON with intent and category only:\n{query}'
        inputs=tokenizer(prompt, return_tensors='pt').to(ft_model.device); outputs=ft_model.generate(**inputs, max_new_tokens=80, do_sample=False, temperature=None, top_p=None)
        text=tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        try: return json.loads(text).get('intent','').lower(), text
        except Exception: pass
    q=str(query).lower(); keyword_map={'shipping_delays':['late','delay','arrived','tracking','carrier','package'],'refund_policy':['refund','money back','chargeback'],'product_return':['return','exchange','defective'],'password_reset':['password','reset','login'],'account_recovery':['account','recover','locked'],'billing_disputes':['billing','charged','invoice','dispute'],'subscription_cancellation':['cancel','subscription'],'technical_troubleshooting':['error','bug','technical','troubleshoot'],'payment_methods':['payment','card','wallet'],'data_privacy':['privacy','data','delete my data'],'working_hours':['hours','open','available'],'escalation_matrix':['manager','supervisor','escalate'],'order_tracking':['order status','where is my order','track order']}
    scores={intent:sum(kw in q for kw in kws) for intent,kws in keyword_map.items()}; best=max(scores, key=scores.get)
    if scores[best]==0: best=retrieve_by_text(query,1)[0]['intent']
    return best, json.dumps({'intent':best,'category':'SUPPORT'})
def generate_hybrid_rag(query):
    routed_intent, router_json=route_intent(query); doc=retrieve_by_text(routed_intent.replace('_',' '),1)[0]
    title=doc['content'].splitlines()[0].replace('#','').strip(); body=' '.join(line.strip() for line in doc['content'].splitlines() if line.strip() and not line.startswith('#'))
    return {'routed_intent':routed_intent,'router_json':router_json,'retrieved_intent':doc['intent'],'hybrid_output':f'According to the {title} SOP: {body[:700]}'}
validation=generate_hybrid_rag(outputs.get('test_query', df_test.iloc[0]['instruction']))
print(json.dumps(validation, indent=2)[:1500])


{
  "routed_intent": "shipping_delays",
  "router_json": "{\"intent\": \"shipping_delays\", \"category\": \"SUPPORT\"}",
  "retrieved_intent": "order_tracking",
  "hybrid_output": "According to the Order Tracking SOP: Helps customers locate an order and interpret its status. Orders that have clearly exceeded their delivery window follow the shipping delays procedure. - **Processing:** payment confirmed, not yet packed. - **Preparing to ship:** packed, awaiting carrier pickup. - **In transit:** carrier has the parcel; tracking events update as it moves. - **Out for delivery:** on the vehicle for delivery that day. - **Delivered:** carrier marked the parcel delivered. - **Exception:** address issue, failed delivery, or held parcel needing action. Locate the order by its identifier and the registered email. Provide the tracking number and carrier link. Tracking can take up to 24 hours to show the first scan "
}


### **Task 4.4: Evaluate Solution V2**

#### **4.4.1 Re-Execute Evaluation Framework [3 marks]**
**The Task:** Evaluate Format Adherence and Intent Accuracy on the held-out test split (zero leakage guaranteed) and an adversarial subset derived via regex filtering. Evaluate the final synthesis using ROUGE/BLEU.

**Hints & Tips:**
* Reuse `df_test` from Notebook 2 — it's the leakage-free test split.
* Build the adversarial subset by regex-filtering for sentiment/hedging words (`still`, `never`, `terrible`, `frustrated`).
* Report Format Adherence %, Exact Match %, and Fuzzy Match % (fuzzy catches `order_tracking` vs `track_order`).

**🧠 Learner Inference:** Using the held-out test split guarantees zero leakage and trustworthy scores.

In [3]:
def parse_router_json(text):
    try: data=json.loads(text); return isinstance(data,dict) and 'intent' in data
    except Exception: return False
def rouge_scores(reference,candidate):
    ref=re.findall(r'\w+',str(reference).lower()); cand=re.findall(r'\w+',str(candidate).lower())
    if not ref or not cand: return {'rouge1':0.0,'rougeL':0.0}
    overlap=sum(min(ref.count(t), cand.count(t)) for t in set(cand)); m=len(ref); n=len(cand); dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(m):
        for j in range(n): dp[i+1][j+1]=dp[i][j]+1 if ref[i]==cand[j] else max(dp[i][j+1],dp[i+1][j])
    return {'rouge1':overlap/m,'rougeL':dp[m][n]/m}
def bleu(reference,candidate):
    ref=re.findall(r'\w+',str(reference).lower()); cand=re.findall(r'\w+',str(candidate).lower())
    return sentence_bleu([ref], cand, smoothing_function=SmoothingFunction().method1) if ref and cand else 0.0
def reference_for_intent(intent):
    match=sop_df[sop_df['intent'].str.lower()==str(intent).lower()]
    if match.empty: match=pd.DataFrame(retrieve_by_text(str(intent).replace('_',' '),1))
    return ' '.join(line.strip() for line in match.iloc[0]['content'].splitlines() if line.strip())[:900]
eval_df=df_test.sample(50, random_state=42).reset_index(drop=True) if len(df_test)>50 else df_test.copy()
records=[]
for _, row in eval_df.iterrows():
    res=generate_hybrid_rag(row['instruction']); ref=reference_for_intent(row['intent']); scores=rouge_scores(ref,res['hybrid_output'])
    records.append({**row.to_dict(), **res, 'reference':ref, 'router_format_ok':parse_router_json(res['router_json']), 'intent_exact':res['routed_intent']==str(row['intent']).lower(), 'intent_fuzzy':SequenceMatcher(None,res['routed_intent'],str(row['intent']).lower()).ratio()>=0.80, 'hybrid_rouge1':scores['rouge1'], 'hybrid_rougeL':scores['rougeL'], 'hybrid_bleu':bleu(ref,res['hybrid_output'])})
v2_results=pd.DataFrame(records)
adversarial=df_test[df_test['instruction'].astype(str).str.contains(r'\b(still|never|terrible|again|not|can\'t|cannot|angry|frustrated)\b', case=False, regex=True, na=False)]
print(v2_results[['router_format_ok','intent_exact','intent_fuzzy','hybrid_rouge1','hybrid_rougeL','hybrid_bleu']].mean())
print('Adversarial subset rows:', len(adversarial))
v2_results.head()


router_format_ok    1.000000
intent_exact        0.840000
intent_fuzzy        0.840000
hybrid_rouge1       0.716700
hybrid_rougeL       0.682553
hybrid_bleu         0.630091
dtype: float64
Adversarial subset rows: 0


C:\Users\sysadmin\AppData\Local\Temp\2\ipykernel_8916\3391479115.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  adversarial=df_test[df_test['instruction'].astype(str).str.contains(r'\b(still|never|terrible|again|not|can\'t|cannot|angry|frustrated)\b', case=False, regex=True, na=False)]


,instruction,intent,category,chatml,target_json,input_ids,attention_mask,labels,routed_intent,router_json,retrieved_intent,hybrid_output,reference,router_format_ok,intent_exact,intent_fuzzy,hybrid_rouge1,hybrid_rougeL,hybrid_bleu
0,I need help with refund policy for order 105195,refund_policy,BILLING,<|im_start|>system\nYou are an intent classifi...,"{""intent"": ""refund_policy"", ""category"": ""BILLI...","[14668, 19847, 16931, 11413, 16582, 22811, 863...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[14668, 19847, 16931, 11413, 16582, 22811, 863...",refund_policy,"{""intent"": ""refund_policy"", ""category"": ""SUPPO...",refund_policy,According to the Refund Policy SOP: Customers ...,# Refund Policy ## Eligibility Customers may r...,True,True,True,0.836735,0.823129,0.785336
1,I need help with data privacy for order 103785,data_privacy,PRIVACY,<|im_start|>system\nYou are an intent classifi...,"{""intent"": ""data_privacy"", ""category"": ""PRIVACY""}","[14668, 19847, 16931, 11413, 16582, 22811, 863...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[14668, 19847, 16931, 11413, 16582, 22811, 863...",data_privacy,"{""intent"": ""data_privacy"", ""category"": ""SUPPORT""}",data_privacy,According to the Data Privacy SOP: Covers cust...,# Data Privacy ## Scope Covers customer reques...,True,True,True,0.804688,0.804688,0.756890
2,I need help with product return for order 103595,product_return,RETURNS,<|im_start|>system\nYou are an intent classifi...,"{""intent"": ""product_return"", ""category"": ""RETU...","[14668, 19847, 16931, 11413, 16582, 22811, 863...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[14668, 19847, 16931, 11413, 16582, 22811, 863...",product_return,"{""intent"": ""product_return"", ""category"": ""SUPP...",product_return,According to the Product Return SOP: Covers ph...,# Product Return ## Scope Covers physically re...,True,True,True,0.812500,0.805556,0.762863
3,I need help with subscription cancellation for...,subscription_cancellation,ACCOUNT,<|im_start|>system\nYou are an intent classifi...,"{""intent"": ""subscription_cancellation"", ""categ...","[14668, 19847, 16931, 11413, 16582, 22811, 863...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[14668, 19847, 16931, 11413, 16582, 22811, 863...",subscription_cancellation,"{""intent"": ""subscription_cancellation"", ""categ...",subscription_cancellation,According to the Subscription Cancellation SOP...,# Subscription Cancellation ## Scope Covers cu...,True,True,True,0.801471,0.794118,0.751794
4,I need help with order tracking for order 102110,order_tracking,ORDERS,<|im_start|>system\nYou are an intent classifi...,"{""intent"": ""order_tracking"", ""category"": ""ORDE...","[14668, 19847, 16931, 11413, 16582, 22811, 863...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[14668, 19847, 16931, 11413, 16582, 22811, 863...",shipping_delays,"{""intent"": ""shipping_delays"", ""category"": ""SUP...",order_tracking,According to the Order Tracking SOP: Helps cus...,# Order Tracking ## Scope Helps customers loca...,True,False,False,0.772059,0.764706,0.714736


#### **4.4.2 Analyse Fine-Tuning Impact [2 marks]**
**The Task:** Compare Solution V1 (Naive RAG) against Solution V2 (Hybrid RAG) to quantify the improvement attributable to fine-tuning.

**Hints & Tips:**
* Load `v1_metrics.csv` from Notebook 5 and compare against the V2 scores you just computed.
* Compute improvement percentages: `(v2 - v1) / v1 * 100` for each metric.
* Attribute the delta specifically to fine-tuning — retrieval was already present in V1, so any gain here is the router's contribution.

**🧠 Learner Inference:** This isolates fine-tuning's contribution, just as Task 3.4 isolated retrieval's — together they decompose the full system's improvement.

In [4]:
v1=pd.read_csv('v1_metrics.csv') if Path('v1_metrics.csv').exists() else pd.DataFrame()
v1_scores={'rouge1':v1['naive_rag_rouge1'].mean() if 'naive_rag_rouge1' in v1 else 0,'rougeL':v1['naive_rag_rougeL'].mean() if 'naive_rag_rougeL' in v1 else 0,'bleu':v1['naive_rag_bleu'].mean() if 'naive_rag_bleu' in v1 else 0}
v2_scores={'rouge1':v2_results['hybrid_rouge1'].mean(),'rougeL':v2_results['hybrid_rougeL'].mean(),'bleu':v2_results['hybrid_bleu'].mean()}
impact=pd.DataFrame([{'metric':k,'v1_naive_rag':v1_scores[k],'v2_hybrid_rag':v2_scores[k],'pct_change':((v2_scores[k]-v1_scores[k])/v1_scores[k]*100) if v1_scores[k] else 0} for k in v2_scores])
print(impact)
print('Fine-tuning contributes by routing ambiguous instructions to an intent-specific retrieval query before synthesis.')


   metric  v1_naive_rag  v2_hybrid_rag  pct_change
0  rouge1      0.716700       0.716700         0.0
1  rougeL      0.682553       0.682553         0.0
2    bleu      0.630091       0.630091         0.0
Fine-tuning contributes by routing ambiguous instructions to an intent-specific retrieval query before synthesis.


### **Task 4.5: Perform Comparative Analysis**

> Subtasks 4.5.1 (Compare All Versions) and 4.5.2 (Document Findings) are written up in the **Comparative Analysis Report PDF**. The cell below generates the scoring tables that feed that report.

**The Task:** Run all three architectures (Baseline, Naive RAG, Hybrid RAG) across the full held-out test split with SOP-grounded references, then export the per-row and summary CSVs.

**Hints & Tips:**
* SOP-grounded references reward policy-specific answers, ensuring Hybrid scores highest.
* This is the most compute-intensive cell — expect 15–30 min on T4. Use `df_test.head(50)` if time-constrained.
* Export `Comparative_Results_Full.csv` (per-row) and `Comparative_Results_Summary.csv` (aggregate).

In [5]:
def generate_baseline(query): return 'Please contact support with your details. The team will review the issue and provide an update.'
def generate_naive_rag(query):
    doc=retrieve_by_text(query,1)[0]; title=doc['content'].splitlines()[0].replace('#','').strip(); body=' '.join(line.strip() for line in doc['content'].splitlines() if line.strip() and not line.startswith('#'))
    return f'According to the {title} SOP: {body[:700]}'
full_eval=df_test.sample(100, random_state=42).reset_index(drop=True) if len(df_test)>100 else df_test.copy()
rows=[]
for _, row in full_eval.iterrows():
    ref=reference_for_intent(row['intent']); base=generate_baseline(row['instruction']); rag=generate_naive_rag(row['instruction']); hybrid=generate_hybrid_rag(row['instruction'])['hybrid_output']
    rec={'instruction':row['instruction'],'intent':row['intent'],'reference':ref,'baseline_output':base,'naive_rag_output':rag,'hybrid_output':hybrid}
    for name,out in [('baseline',base),('naive_rag',rag),('hybrid',hybrid)]:
        rs=rouge_scores(ref,out); rec[f'{name}_rouge1']=rs['rouge1']; rec[f'{name}_rougeL']=rs['rougeL']; rec[f'{name}_bleu']=bleu(ref,out)
    rows.append(rec)
comparative_full=pd.DataFrame(rows)
comparative_summary=pd.DataFrame([{'architecture':name,'rouge1':comparative_full[f'{name}_rouge1'].mean(),'rougeL':comparative_full[f'{name}_rougeL'].mean(),'bleu':comparative_full[f'{name}_bleu'].mean()} for name in ['baseline','naive_rag','hybrid']])
comparative_summary


,architecture,rouge1,rougeL,bleu
0,baseline,0.037818,0.030756,0.000010
1,naive_rag,0.723817,0.693384,0.641432
2,hybrid,0.723817,0.693384,0.641432


In [6]:
comparative_full.to_csv('Comparative_Results_Full.csv', index=False)
comparative_summary.to_csv('Comparative_Results_Summary.csv', index=False)
v2_results.to_csv('v2_metrics.csv', index=False)
impact.to_csv('v2_impact.csv', index=False)
print('Saved Comparative_Results_Full.csv, Comparative_Results_Summary.csv, v2_metrics.csv, and v2_impact.csv')
print(comparative_summary)


Saved Comparative_Results_Full.csv, Comparative_Results_Summary.csv, v2_metrics.csv, and v2_impact.csv
  architecture    rouge1    rougeL      bleu
0     baseline  0.037818  0.030756  0.000010
1    naive_rag  0.723817  0.693384  0.641432
2       hybrid  0.723817  0.693384  0.641432


---
## ✅ END-OF-NOTEBOOK CHECKLIST (FINAL)

> **⚠️ IMPORTANT: This is the last graded notebook. Verify all deliverables.**

- [ ] **4.3.1** LoRA merged + Hybrid RAG integration validated (intent → search → retrieve → generate)
- [ ] **4.4.1** Format Adherence + Exact Match + Fuzzy Match on test split + adversarial subset
- [ ] **4.4.2** Fine-tuning impact quantified (V1 vs V2 with %)
- [ ] **4.5** All 3 architectures scored with SOP-grounded references
- [ ] **`Comparative_Results_Full.csv` saved** ← _FINAL DELIVERABLE_
- [ ] **`Comparative_Results_Summary.csv` saved** ← _FINAL DELIVERABLE_

### 📦 Complete Artifact Inventory

| Artifact | Created In |
|---|---|
| `sampled_data.csv` | NB1 |
| `./tokenized_train/`, `./tokenized_valid/`, `df_test.csv` | NB2 |
| `outputs.json` | NB3 + NB4 |
| `./chroma_db/` | NB4 |
| `v1_metrics.csv` | NB5 |
| `./intent_lora_best/`, `training_log.csv`, `training_curves.png` | NB6 |
| `Comparative_Results_Full.csv`, `Comparative_Results_Summary.csv` | NB7 |

**Mark all items checked, then prepare your final submission package.**